# M12 Lab — Local LLM Mastery

**Datasets:** your own notebooks + CSVs &nbsp;|&nbsp; **Focus:** Ollama, synthetic data, RAG, privacy-aware local AI


In [ ]:
import requests

r = requests.get("http://localhost:11434/api/tags", timeout=10)
print("status:", r.status_code)
print(r.json())


## L12.2 — Basic Ollama API call


In [ ]:
import requests

payload = {
    "model": "gemma4n",
    "messages": [{"role": "user", "content": "Explain in 2 sentences why local LLMs matter for data science."}],
    "stream": False,
}

response = requests.post("http://localhost:11434/api/chat", json=payload, timeout=60)
response.raise_for_status()
print(response.json()["message"]["content"])


## L12.2 — Streaming output


In [ ]:
import json
import requests

payload = {
    "model": "gemma4n",
    "messages": [{"role": "user", "content": "List 3 privacy benefits of a local LLM for students."}],
    "stream": True,
}

with requests.post("http://localhost:11434/api/chat", json=payload, stream=True, timeout=60) as r:
    r.raise_for_status()
    for line in r.iter_lines():
        if not line:
            continue
        chunk = json.loads(line.decode("utf-8"))
        if "message" in chunk and "content" in chunk["message"]:
            print(chunk["message"]["content"], end="")
        if chunk.get("done"):
            break
print()


## L12.3 — Structured JSON output


In [ ]:
import json
import requests

prompt = """
Return ONLY a valid JSON array with 3 rows.
Each row must contain: hypothesis, metric, caution.
Context: churn dataset with columns tenure, monthly_spend, churn.
"""

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "gemma4n",
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    },
    timeout=60,
)
response.raise_for_status()
raw = response.json()["message"]["content"]
hypotheses = json.loads(raw)
print(hypotheses)


## L12.4 — Synthetic data generation [🔮 ai-generated]


In [ ]:
import requests, json, pandas as pd

def ask_gemma(prompt: str, model: str = "gemma4n") -> str:
    r = requests.post(
        "http://localhost:11434/api/chat",
        json={"model": model, "messages": [{"role": "user", "content": prompt}], "stream": False}
    )
    return r.json()["message"]["content"]

schema_prompt = """
Generate 20 rows of realistic e-commerce order data as a JSON array.
Each row: {"order_id": int, "customer_age": int, "product_category": str,
           "order_value": float, "days_to_delivery": int, "returned": bool}
Constraints: age 18-75, value 5.00-500.00, delivery 1-14 days.
Return ONLY the JSON array, no explanation.
"""

raw = ask_gemma(schema_prompt)
df = pd.DataFrame(json.loads(raw))
print(df.dtypes)
df.describe()


## L12.4 — Validate synthetic data [AI-VERIFY]


In [ ]:
import pandera as pa
from pandera import Check, Column, DataFrameSchema

schema = DataFrameSchema(
    {
        "order_id": Column(int),
        "customer_age": Column(int, checks=Check.in_range(18, 75)),
        "product_category": Column(str),
        "order_value": Column(float, checks=Check.in_range(5.0, 500.0)),
        "days_to_delivery": Column(int, checks=Check.in_range(1, 14)),
        "returned": Column(bool),
    }
)

validated_df = schema.validate(df)
print(validated_df.head())


## L12.5 — RAG pipeline


In [ ]:
        import importlib.util
        import json
        from pathlib import Path

        missing = [
            pkg for pkg in ["sentence_transformers", "faiss", "numpy", "pandas"]
            if importlib.util.find_spec(pkg) is None
        ]
        if missing:
            raise ImportError(f"Missing packages for this cell: {missing}")

        import faiss
        import numpy as np
        import pandas as pd
        import requests
        from sentence_transformers import SentenceTransformer

        def load_documents(folder: str = "."):
            docs = []
            for path in Path(folder).glob("*.ipynb"):
                nb = json.loads(path.read_text(encoding="utf-8"))
                docs.append({"source": path.name, "text": "
".join("".join(c.get("source", [])) for c in nb.get("cells", []))})
            for path in Path(folder).glob("*.csv"):
                frame = pd.read_csv(path)
                docs.append({
                    "source": path.name,
                    "text": f"Columns: {list(frame.columns)}
Shape: {frame.shape}
Preview:
{frame.head(5).to_csv(index=False)}",
                })
            return docs

        def chunk_text(text: str, size: int = 400, overlap: int = 80):
            chunks = []
            start = 0
            while start < len(text):
                chunks.append(text[start:start + size])
                start += size - overlap
            return chunks

        docs = load_documents(".")
        chunk_records = []
        for doc in docs:
            for chunk in chunk_text(doc["text"]):
                chunk_records.append({"source": doc["source"], "chunk": chunk})

        embedder = SentenceTransformer("all-MiniLM-L6-v2")
        embeddings = np.asarray(embedder.encode([r["chunk"] for r in chunk_records], normalize_embeddings=True), dtype="float32")
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)

        query = "Which local files mention confidence intervals, p-values, or effect size?"
        query_vec = np.asarray(embedder.encode([query], normalize_embeddings=True), dtype="float32")
        _, idxs = index.search(query_vec, k=min(3, len(chunk_records)))
        retrieved = [chunk_records[i] for i in idxs[0]]
        context = "

".join(f"[{r['source']}]
{r['chunk']}" for r in retrieved)

        answer = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": "gemma4n",
                "messages": [{
                    "role": "user",
                    "content": f"Use only the retrieved context below. If unsupported, say so.

Context:
{context}

Question: {query}",
                }],
                "stream": False,
            },
            timeout=120,
        )
        answer.raise_for_status()
        print("retrieved sources:", [r["source"] for r in retrieved])
        print(answer.json()["message"]["content"])


## L12.7 [AI-OFF] — Privacy design exercise


In a new markdown cell below this one, design a privacy-preserving local AI workflow for a data-science project.

Your answer must classify each stage as **AI-assisted**, **AI-generated**, or **AI-independent**.
Include at least these stages:
1. loading data,
2. retrieving notebook context,
3. generating a chart narrative,
4. verifying the final result,
5. logging the interaction in `AI_USE.md`.

Then identify at least 3 leakage risks and 3 mitigations.
Do this **without** model assistance.
